# SVM model using Count Vectors for Unigram+Bigram

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import cross_val_score, StratifiedKFold
from lime.lime_text import LimeTextExplainer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [2]:
# Load the dataset
df = pd.read_csv('yelp_sample_20k.csv')

In [3]:
df

,text,stars
0,We really enjoyed their use of homemade ingred...,5.0
1,We moved to Sarasota and upon many recommendat...,5.0
2,Very disappointed in today's meal. The wife an...,2.0
3,"I wish I could reward the store w 5 stars, pro...",4.0
4,"Came here for brunch yesterday, it goes till 3...",5.0
...,...,...
19995,One of the worst before dinner experiences I h...,1.0
19996,This place is magical. It's perfect for weddin...,5.0
19997,to the person that hit my brand new car yester...,2.0
19998,Lilly's is one of the best places to go for lu...,5.0


In [4]:
# Map star ratings to sentiment categories
def assign_sentiment(stars):
    if stars >= 4:
        return 'Positive'
    elif stars <= 2:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['stars'].apply(assign_sentiment)

In [5]:
df.shape

(20000, 3)

In [6]:
df.head()

,text,stars,Sentiment
0,We really enjoyed their use of homemade ingred...,5.0,Positive
1,We moved to Sarasota and upon many recommendat...,5.0,Positive
2,Very disappointed in today's meal. The wife an...,2.0,Negative
3,"I wish I could reward the store w 5 stars, pro...",4.0,Positive
4,"Came here for brunch yesterday, it goes till 3...",5.0,Positive


In [7]:
# Preprocessing: Convert text to lowercase, remove stopwords, punctuations, digits, and apply lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove digits
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuations
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

df['cleaned_text'] = df['text'].apply(preprocess_text)

In [8]:
# Drop unnecessary columns and handle missing values
df = df[['cleaned_text','stars','Sentiment']].dropna()

In [9]:
df.head()

,cleaned_text,stars,Sentiment
0,really enjoyed use homemade ingredient really ...,5.0,Positive
1,moved sarasota upon many recommendation decide...,5.0,Positive
2,disappointed today meal wife got pasta dish wa...,2.0,Negative
3,wish could reward store w star product marketi...,4.0,Positive
4,came brunch yesterday go till pm great late ri...,5.0,Positive


In [10]:
# Split the dataset
X = df['cleaned_text']
y = df['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [11]:
# Feature extraction using Count Vectors for Unigram+Bigram with max_features
count_vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english', max_features=5000)  # Select top 5000 features
X_train_cv = count_vectorizer.fit_transform(X_train)
X_test_cv = count_vectorizer.transform(X_test)

In [12]:
# SVM using Count Vectors for Unigram+Bigram
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_cv, y_train)
svm_preds = svm_model.predict(X_test_cv)
svm_probs = svm_model.decision_function(X_test_cv)

In [13]:
# Train the SVM model with default parameters
start_time = time.time()
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_cv, y_train)
end_time = time.time()

training_time = end_time - start_time
print(f"Training Time: {training_time:.2f} seconds")

Training Time: 346.27 seconds


In [14]:
# Predictions and decision scores
svm_preds = svm_model.predict(X_test_cv)
svm_probs = svm_model.decision_function(X_test_cv)

In [15]:
# Evaluation Metrics
print("SVM with Unigram+Bigram (Count Vectors):\n", classification_report(y_test, svm_preds))

SVM with Unigram+Bigram (Count Vectors):
               precision    recall  f1-score   support

    Negative       0.72      0.78      0.75      1380
     Neutral       0.28      0.27      0.28       565
    Positive       0.90      0.88      0.89      4055

    accuracy                           0.80      6000
   macro avg       0.63      0.64      0.64      6000
weighted avg       0.80      0.80      0.80      6000

